In [1]:
from statsmodels.tsa.api import VAR
import pandas as pd
import numpy as np

In [2]:
mat_list = ["0Y1M", "0Y3M", "0Y6M", "1Y", "2Y", "3Y", "5Y", "7Y", "10Y", "20Y", "30Y"]

df = pd.read_csv("../../data/FRB_H15.csv").dropna()
df.columns = ["Date", *mat_list]
df["Date"] = pd.to_datetime(df["Date"])
df.set_index("Date", inplace=True)

In [3]:
# Estimate the three Nelson-Siegel beta factors for each yield curve.
maturities = [1/12, 3/12, 6/12, 1, 2, 3, 5, 7, 10, 20, 30]
lambda_ns = 0.0609
loadings = np.array([
    [
        1,
        (1 - np.exp(-lambda_ns * maturity)) / (lambda_ns * maturity),
        ((1 - np.exp(-lambda_ns * maturity)) / (lambda_ns * maturity))
        - np.exp(-lambda_ns * maturity),
    ]
    for maturity in maturities
])

beta_values = [
    np.linalg.lstsq(loadings, yields, rcond=None)[0]
    for yields in df.to_numpy()
]
betas = pd.DataFrame(
    beta_values,
    index=df.index,
    columns=["Beta 1", "Beta 2", "Beta 3"],
)

In [4]:
# Refit a five-lag VAR for exactly 20 expanding, one-day-ahead rolling forecasts.
rolling_windows = 20
forecast_indexes = range(len(betas) - rolling_windows, len(betas))
lag = 5

def rolling_metrics(errors, predicted_changes, actual_changes):
    rmse_basis_points = 100 * np.sqrt(np.mean(np.asarray(errors) ** 2))
    directional_accuracy = 100 * np.mean(
        np.sign(predicted_changes) == np.sign(actual_changes)
    )
    return rmse_basis_points, directional_accuracy

errors = np.zeros((rolling_windows, 3))
predicted_changes = np.zeros((rolling_windows, 3))
actual_changes = np.zeros((rolling_windows, 3))

for window_index, forecast_index in enumerate(forecast_indexes):
    history = betas.iloc[:forecast_index]
    var_result = VAR(history.reset_index(drop=True)).fit(lag)

    current = history.iloc[-1].to_numpy()
    predicted = var_result.forecast(
        history.to_numpy()[-var_result.k_ar:],
        steps=1,
    )[0]
    actual = betas.iloc[forecast_index].to_numpy()

    errors[window_index] = predicted - actual
    predicted_changes[window_index] = predicted - current
    actual_changes[window_index] = actual - current

beta_metrics = [
    rolling_metrics(
        errors[:, beta_index],
        predicted_changes[:, beta_index],
        actual_changes[:, beta_index],
    )
    for beta_index in range(3)
]

In [5]:
print(f"Rolling windows: {rolling_windows}")
for beta, (beta_rmse, directional_accuracy) in enumerate(beta_metrics, start=1):
    print(f"  Beta: {beta}")
    print(
        f"  Forecast 1 day, RMSE: {beta_rmse} basis points, "
        f"Directional Accuracy: {directional_accuracy}%"
    )

Rolling windows: 20
  Beta: 1
  Forecast 1 day, RMSE: 26.202326835039454 basis points, Directional Accuracy: 40.0%
  Beta: 2
  Forecast 1 day, RMSE: 27.31001564244428 basis points, Directional Accuracy: 35.0%
  Beta: 3
  Forecast 1 day, RMSE: 50.12794570263415 basis points, Directional Accuracy: 50.0%
